<a href="https://colab.research.google.com/github/Serx17/compliance-checker-230fz/blob/main/notebooks/02_rag_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Тихая установка с подавлением конфликтов
import sys
import warnings
warnings.filterwarnings("ignore")

!pip install chromadb sentence-transformers --quiet --disable-pip-version-check 2>/dev/null

# 2. Быстрая проверка импорта
try:
    import chromadb
    from sentence_transformers import SentenceTransformer
    print("✅ Библиотеки успешно загружены")

    # Тест модели (загружаем только архитектуру, без скачивания весов)
    # Веса скачаются в следующей ячейке при первом encode()
    print("⏳ Проверка доступности sentence-transformers...")
    _ = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
    print("✅ Модель инициализирована. Готово к индексации.")

except Exception as e:
    print(f"❌ Ошибка инициализации: {e}")
    print("💡 Попробуй перезапустить runtime (Runtime -> Restart session) и запустить снова.")

✅ Библиотеки успешно загружены
⏳ Проверка доступности sentence-transformers...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Модель инициализирована. Готово к индексации.


In [5]:
import chromadb
from sentence_transformers import SentenceTransformer
import os
import shutil

# ==========================================
# 1. НАСТРОЙКИ (Исправлен путь для Colab)
# ==========================================
# Используем абсолютный путь /content/, который гарантированно доступен для записи
DB_PATH = "/content/chroma_db_230fz"

if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)
    print(f"🧹 Очищена старая база: {DB_PATH}")

# Явно создаём директорию до инициализации ChromaDB
os.makedirs(DB_PATH, exist_ok=True)

# ==========================================
# 2. ДАННЫЕ (Чанки закона 230-ФЗ)
# ==========================================
LAW_CHUNKS = [
    {"id": "chunk_1", "source": "Статья 6, ч. 1", "content": "Запрещаются действия по возврату просроченной задолженности, которые причиняют вред жизни или здоровью должника, повреждают его имущество или оказывают на него психологическое давление. Запрещается вводить должника в заблуждение."},
    {"id": "chunk_2", "source": "Статья 7, ч. 5", "content": "Запрещается взаимодействие с должником в рабочие дни с 22 до 8 часов по местному времени, а также в выходные и нерабочие праздничные дни с 20 до 9 часов."},
    {"id": "chunk_3", "source": "Статья 7, ч. 4", "content": "Кредитор или лицо, действующее от его имени, не вправе осуществлять взаимодействие с должником чаще двух раз в течение недели и чаще восьми раз в течение месяца."},
    {"id": "chunk_4", "source": "Статья 8", "content": "При взаимодействии с должником запрещается раскрывать третьим лицам сведения о наличии просроченной задолженности, а также иную информацию о должнике."}
]

# ==========================================
# 3. ЗАГРУЗКА МОДЕЛИ И ИНДЕКСАЦИЯ
# ==========================================
print("⏳ Загрузка модели эмбеддингов (CPU)...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device="cpu")

documents = [chunk["content"] for chunk in LAW_CHUNKS]
metadatas = [{"source": chunk["source"]} for chunk in LAW_CHUNKS]
ids = [chunk["id"] for chunk in LAW_CHUNKS]

print("🧮 Генерация векторов (Embeddings)...")
embeddings = embedding_model.encode(documents, show_progress_bar=False).tolist()

# ==========================================
# 4. СОХРАНЕНИЕ В БАЗУ
# ==========================================
print(f"💾 Создание базы в {DB_PATH}...")
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection(name="230fz_law")

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings
)

print("✅ Готово! База создана и проиндексирована.")
print(f"📂 Индексировано статей: {collection.count()}")

⏳ Загрузка модели эмбеддингов (CPU)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🧮 Генерация векторов (Embeddings)...
💾 Создание базы в /content/chroma_db_230fz...
✅ Готово! База создана и проиндексирована.
📂 Индексировано статей: 4


In [7]:
import chromadb
from sentence_transformers import SentenceTransformer
import os
import shutil

# ==========================================
# 1. НАСТРОЙКИ И ОЧИСТКА
# ==========================================
DB_PATH = "/content/chroma_db_230fz_v2" # Новая папка, чтобы не смешивать
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)
os.makedirs(DB_PATH, exist_ok=True)

# ==========================================
# 2. ДАННЫЕ (С ОБАГОЩЕНИЕМ / SYNONYM EXPANSION)
# ==========================================
# Мы добавляем блок "Контекст", который не является частью закона,
# но помогает модели связать юридические термины с бытовыми словами.
LAW_CHUNKS = [
    {
        "id": "chunk_1",
        "source": "Статья 6, ч. 1",
        "content": "Запрещаются действия по возврату просроченной задолженности, которые причиняют вред жизни или здоровью должника, повреждают его имущество или оказывают на него психологическое давление. Запрещается вводить должника в заблуждение. "
                   "Контекст поиска: угрозы физической расправой, угрозы испортить кредитную историю, угрозы уволить с работы, шантаж, агрессия, крик."
    },
    {
        "id": "chunk_2",
        "source": "Статья 7, ч. 5",
        "content": "Запрещается взаимодействие с должником в рабочие дни с 22 до 8 часов по местному времени, а также в выходные и нерабочие праздничные дни с 20 до 9 часов. "
                   "Контекст поиска: ночное время, ночной звонок, звонок вечером, раннее утро, выходные, праздники."
    },
    {
        "id": "chunk_3",
        "source": "Статья 7, ч. 4",
        "content": "Кредитор или лицо, действующее от его имени, не вправе осуществлять взаимодействие с должником чаще двух раз в течение недели и чаще восьми раз в течение месяца. "
                   "Контекст поиска: частые звонки, спам, сколько раз можно звонить, лимит звонков, постоянные смс."
    },
    {
        "id": "chunk_4",
        "source": "Статья 8",
        "content": "При взаимодействии с должником запрещается раскрывать третьим лицам сведения о наличии просроченной задолженности, а также иную информацию о должнике. "
                   "Контекст поиска: третьи лица, начальник, коллеги, соседи, родственники, друзья, кредитная история, работодатель, подъемезд."
    }
]

# ==========================================
# 3. ИНДЕКСАЦИЯ
# ==========================================
print("⏳ Загрузка модели и создание новой базы с обогащенными данными...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device="cpu")

documents = [chunk["content"] for chunk in LAW_CHUNKS]
metadatas = [{"source": chunk["source"]} for chunk in LAW_CHUNKS]
ids = [chunk["id"] for chunk in LAW_CHUNKS]

embeddings = embedding_model.encode(documents, show_progress_bar=False).tolist()

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection(name="230fz_law_rich")

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=embeddings
)
print("✅ База обновлена. Теперь модель понимает контекст.\n")

# ==========================================
# 4. ПОВТОРНЫЙ ТЕСТ (RETRIEVAL)
# ==========================================
def find_relevant_articles(user_text: str, top_k: int = 2):
    query_embedding = embedding_model.encode([user_text]).tolist()
    return collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

test_queries = [
    "Рассказали моему начальнику про долг",
    "Звонили в 11 вечера",
    "Сколько раз можно звонить?"
]

print("🔍 ТЕСТ ПОСЛЕ ОБАГОЩЕНИЯ ДАННЫХ")
print("=" * 50)

for q in test_queries:
    print(f"\n❓ Запрос: '{q}'")
    res = find_relevant_articles(q)

    # Берем самый релевантный результат (индекс 0)
    best_doc = res['documents'][0][0]
    best_source = res['metadatas'][0][0]['source']

    print(f"   🎯 Найдено: {best_source}")
    # Показываем только начало текста, чтобы было читаемо
    print(f"   📝 Текст: {best_doc[:50]}...")
    print("-" * 30)

⏳ Загрузка модели и создание новой базы с обогащенными данными...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ База обновлена. Теперь модель понимает контекст.

🔍 ТЕСТ ПОСЛЕ ОБАГОЩЕНИЯ ДАННЫХ

❓ Запрос: 'Рассказали моему начальнику про долг'
   🎯 Найдено: Статья 8
   📝 Текст: При взаимодействии с должником запрещается раскрыв...
------------------------------

❓ Запрос: 'Звонили в 11 вечера'
   🎯 Найдено: Статья 7, ч. 5
   📝 Текст: Запрещается взаимодействие с должником в рабочие д...
------------------------------

❓ Запрос: 'Сколько раз можно звонить?'
   🎯 Найдено: Статья 7, ч. 4
   📝 Текст: Кредитор или лицо, действующее от его имени, не вп...
------------------------------
